In [29]:
import pandas as pd
import numpy as np

seq_regions = ["SEQ_H1", "SEQ_H2", "SEQ_L1", "SEQ_L2", "SEQ_L3"]
cf_regions = ["CF_H1", "CF_H2", "CF_L1", "CF_L2", "CF_L3"]
len_regions = ["LEN_H1", "LEN_H2", "LEN_L1", "LEN_L2", "LEN_L3"]

df = (
    pd.read_csv("data/ab_ag_scalop.tsv", sep="\t")
    #.dropna(subset = seq_regions)
    #.dropna(subset = cf_regions)
    .drop_duplicates(subset = seq_regions) 
)

antigen_counts = df["antigen_name"].value_counts() # Tabelle aus antigen_names und ihren Häufigkeiten in der Spalte antigen_name
df = df[df["antigen_name"].isin(antigen_counts[antigen_counts >= 5].index)] # Behält nur Zeilen, deren antigen_name mindestens 5-mal vorkommt

In [30]:
for cf_region, seq_region in zip(cf_regions, seq_regions):    
    print(f"Cluster in {cf_region}: {sorted(df[cf_region].astype(str).unique().tolist())}")
    print(f"Längen in {cf_region}: {sorted(df[seq_region].astype(str).map(len).unique().tolist())}\n")
    

Cluster in CF_H1: ['H1-7-A', 'H1-7-B', 'H1-7-C', 'H1-7-D', 'H1-8-A', 'H1-8-B', 'H1-9-A', 'H1-9-B', 'nan']
Längen in CF_H1: [4, 5, 6, 7, 8, 9, 12, 13, 16]

Cluster in CF_H2: ['H2-5-A', 'H2-6-A', 'H2-6-B', 'H2-6-C', 'H2-6-D', 'H2-6-E', 'H2-8-A', 'nan']
Längen in CF_H2: [4, 5, 6, 7, 8, 10, 11, 15, 16]

Cluster in CF_L1: ['L1-10-A', 'L1-11-A', 'L1-11-B', 'L1-12-A', 'L1-12-B', 'L1-12-C', 'L1-13-A', 'L1-13-B', 'L1-13-C', 'L1-14-A', 'L1-14-B', 'L1-14-C', 'L1-15-A', 'L1-16,17-A', 'nan']
Längen in CF_L1: [3, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17]

Cluster in CF_L2: ['L2-7-A', 'nan']
Längen in CF_L2: [3, 5, 7, 11, 12]

Cluster in CF_L3: ['L3-10,11-A', 'L3-10-A', 'L3-10-B', 'L3-11-A', 'L3-5-A', 'L3-8-A', 'L3-9,10-A', 'L3-9-A', 'nan']
Längen in CF_L3: [5, 6, 7, 8, 9, 10, 11, 12, 13]



In [31]:
# Before clustering, Scalop groups CDR-sequences by length. However, some sequences in our dataset have a length, for which Scalop cannot assign a cluster.
# These sequences are unusually long or short (e.g. für CDR-H1: lengths 4, 5, 6, 12, 13, 16 do not yield any clusters, lenghts 7 and 8 do yield clusters) and are rare in our dataset as they are outliers.
# Proportion tests will be conducted only on length-groups, for which Scalop can assign clusters. Sequences in those length-groups, which were not assigned a cluster (nan) will be considered as cluster "others"
# in their respective length-group. Length groups, that did not yield any clusters will be discarded and not analyzed further.

In [ ]:
sorted_dfs = {}

for seq_region, cf_region, len_region in list(zip(seq_regions, cf_regions, len_regions))[0]:
    
    # Neue Spalte für die Sequenzlänge
    df[len_region] = df[seq_region].str.len()

    # Ersetze fehlende Cluster durch "others"
    df[cf_region] = df[cf_region].fillna("others")
    
    # Sortiere nach Länge und Cluster
    sorted_df = df.sort_values(by=[len_region, cf_region])
    
    # Speichere in dict
    sorted_dfs[seq_region] = sorted_df[["pdb", "antigen_name", "antigen_species", "Hchain", "Lchain", "model", seq_region, len_region, cf_region]]

sorted_df = sorted_dfs["SEQ_L1"]